# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [27]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [28]:
# loading the dataset
data = pd.read_csv('AviationData.csv', encoding='Latin-1')
# Checking the first few rows of the dataset
data.head()

<ipython-input-28-e23bc67ecce5>:2: DtypeWarning: Columns (6,7,28) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('AviationData.csv', encoding='Latin-1')


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


In [29]:
# Checking basic information about the dataset
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  object 
 1   Investigation.Type      88889 non-null  object 
 2   Accident.Number         88889 non-null  object 
 3   Event.Date              88889 non-null  object 
 4   Location                88837 non-null  object 
 5   Country                 88663 non-null  object 
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50249 non-null  object 
 9   Airport.Name            52790 non-null  object 
 10  Injury.Severity         87889 non-null  object 
 11  Aircraft.damage         85695 non-null  object 
 12  Aircraft.Category       32287 non-null  object 
 13  Registration.Number     87572 non-null  object 
 14  Make                    88826 non-null

In [30]:
# Cheching for statistical summary
data.describe()

,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


In [31]:
# Checking for nmissing values
data.isnull().sum()

Event.Id                      0
Investigation.Type            0
Accident.Number               0
Event.Date                    0
Location                     52
Country                     226
Latitude                  54507
Longitude                 54516
Airport.Code              38640
Airport.Name              36099
Injury.Severity            1000
Aircraft.damage            3194
Aircraft.Category         56602
Registration.Number        1317
Make                         63
Model                        92
Amateur.Built               102
Number.of.Engines          6084
Engine.Type                7077
FAR.Description           56866
Schedule                  76307
Purpose.of.flight          6192
Air.carrier               72241
Total.Fatal.Injuries      11401
Total.Serious.Injuries    12510
Total.Minor.Injuries      11933
Total.Uninjured            5912
Weather.Condition          4492
Broad.phase.of.flight     27165
Report.Status              6381
Publication.Date          13771
dtype: i

## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [32]:
# Filter the dataset for events that occurred on or after January 1, 1983
data['Event.Date'] = pd.to_datetime(data['Event.Date'], errors='coerce')

data = data[data['Event.Date'] >= '1983-01-01']
data.shape

(85289, 31)

In [33]:
# Standardize the column names by converting them to lowercase and replacing spaces with underscores
data.columns = data.columns.str.lower().str.replace('.', '_')   
data.columns

<ipython-input-33-c503bf0d3b0d>:2: FutureWarning: The default value of regex will change from True to False in a future version. In addition, single character regular expressions will *not* be treated as literal strings when regex=True.
  data.columns = data.columns.str.lower().str.replace('.', '_')


Index(['event_id', 'investigation_type', 'accident_number', 'event_date',
       'location', 'country', 'latitude', 'longitude', 'airport_code',
       'airport_name', 'injury_severity', 'aircraft_damage',
       'aircraft_category', 'registration_number', 'make', 'model',
       'amateur_built', 'number_of_engines', 'engine_type', 'far_description',
       'schedule', 'purpose_of_flight', 'air_carrier', 'total_fatal_injuries',
       'total_serious_injuries', 'total_minor_injuries', 'total_uninjured',
       'weather_condition', 'broad_phase_of_flight', 'report_status',
       'publication_date'],
      dtype='object')

In [34]:
# Handling missing values
# Check missing values again
data.isna().sum().sort_values(ascending=False).head(10)

schedule                 73244
air_carrier              68876
far_description          56830
aircraft_category        56566
longitude                50919
latitude                 50910
airport_code             36737
airport_name             34689
broad_phase_of_flight    27163
publication_date         13769
dtype: int64

In [35]:
# Dropping columns with many missing values
columns_to_drop = ['air_carrier', 'far_description', 'report_status', 'publication_date', 'longitude', 'latitude', 'schedule']
data = data.drop(columns=columns_to_drop)
data.isna().sum().sort_values(ascending=False).head(10)

aircraft_category         56566
airport_code              36737
airport_name              34689
broad_phase_of_flight     27163
total_serious_injuries    12481
total_minor_injuries      11905
total_fatal_injuries      11376
engine_type                7076
purpose_of_flight          6187
number_of_engines          6083
dtype: int64

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [38]:
injury_cols = [
    'total_fatal_injuries',
    'total_serious_injuries',
    'total_minor_injuries',
    'total_uninjured'
]

for col in injury_cols:
    data[col] = pd.to_numeric(data[col], errors='coerce').fillna(0)

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [39]:
# Creating a new column 'is_destroyed' to check wheather the aircraft was destroyed or not
data['is_destroyed'] = data['aircraft_damage'].str.lower().str.contains("destroyed", na=False)
data.columns

Index(['event_id', 'investigation_type', 'accident_number', 'event_date',
       'location', 'country', 'airport_code', 'airport_name',
       'injury_severity', 'aircraft_damage', 'aircraft_category',
       'registration_number', 'make', 'model', 'amateur_built',
       'number_of_engines', 'engine_type', 'purpose_of_flight',
       'total_fatal_injuries', 'total_serious_injuries',
       'total_minor_injuries', 'total_uninjured', 'weather_condition',
       'broad_phase_of_flight', 'is_destroyed'],
      dtype='object')

In [41]:
data['is_destroyed'].value_counts()

False    67714
True     17575
Name: is_destroyed, dtype: int64

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [42]:
# Convert make column to lower cases
data['make'].str.lower().value_counts().head(10)

cessna      25814
piper       14142
beech        5109
boeing       2696
bell         2610
mooney       1278
robinson     1207
grumman      1068
bellanca      983
hughes        879
Name: make, dtype: int64

In [47]:
# Number of unique makes in the dataset
make_counts = data['make'].value_counts()

make_counts.head(20)

Cessna               20891
Piper                11301
CESSNA                4919
Beech                 4067
PIPER                 2840
Bell                  2022
Boeing                1544
BOEING                1145
BEECH                 1041
Mooney                1036
Grumman                990
Robinson               923
Bellanca               824
Hughes                 742
Schweizer              612
BELL                   588
Air Tractor            577
Mcdonnell Douglas      521
Aeronca                458
Maule                  428
Name: make, dtype: int64

In [49]:
# Filter through makes with 50 and over accidents
valid_makes = make_counts[make_counts >= 50].index

data = data[data['make'].isin(valid_makes)]

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [43]:
# Checking for null values in the model column
data['model'].isna().sum()

78

In [44]:
# Removing null values in the model column
data = data.dropna(subset=['model'])
data['model'].isna().sum()

0

In [45]:
# Convert values in the model column to lower string
data['model'] = data['model'].str.lower()
# Checking the most common models in the dataset
data['model'].str.lower().value_counts().head(10)

<ipython-input-45-e15b29d15015>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['model'] = data['model'].str.lower()


152          2232
172          1657
172n         1097
pa-28-140     869
172m          760
150           752
172p          665
182           618
180           596
pa-18-150     557
Name: model, dtype: int64

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [50]:
# Cleaning the engine_type column
data['engine_type'] = data['engine_type'].str.lower()
data['engine_type'].value_counts().head(10)

reciprocating      54835
turbo shaft         2907
turbo prop          2668
turbo fan           2081
unknown             1558
turbo jet            524
geared turbofan       11
none                   2
lr                     1
unk                    1
Name: engine_type, dtype: int64

In [51]:
# Cleaning the weather_condition column
data['weather_condition'] = data['weather_condition'].str.lower()
data['weather_condition'].value_counts().head(10)

vmc    60045
imc     5205
unk      957
Name: weather_condition, dtype: int64

In [52]:
# Cleaning the number_of_engines column
data['number_of_engines'] = pd.to_numeric(data['number_of_engines'], errors='coerce')
data['number_of_engines'].value_counts().head(10)

1.0    54177
2.0     9354
0.0      674
3.0      430
4.0      387
8.0        1
Name: number_of_engines, dtype: int64

In [53]:
# Cleaning the purpose of flight column
data['purpose_of_flight'] = data['purpose_of_flight'].str.lower()
data['purpose_of_flight'].value_counts().head(10)

personal              36708
instructional          9325
unknown                5540
aerial application     4076
business               3409
positioning            1390
other work use         1040
aerial observation      696
public aircraft         661
ferry                   638
Name: purpose_of_flight, dtype: int64

In [54]:
# Cleaning the broad_phase_of_flight column
data['broad_phase_of_flight'] = data['broad_phase_of_flight'].str.lower()
data['broad_phase_of_flight'].value_counts().head(10)

landing        13044
takeoff         9809
cruise          8407
maneuvering     6304
approach        5218
taxi            1661
climb           1651
descent         1560
go-around       1191
standing         831
Name: broad_phase_of_flight, dtype: int64

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [56]:
# Removing columns with too many missing values
data.isna().sum().sort_values(ascending=False).head(10)

aircraft_category        49110
airport_code             30359
airport_name             28509
broad_phase_of_flight    19603
engine_type               5172
purpose_of_flight         5059
number_of_engines         4737
weather_condition         3553
aircraft_damage           2585
registration_number       1163
dtype: int64

In [57]:
# REmoving the registration_number column due to too many missing values
data = data.drop(columns=['registration_number'])


### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [58]:
data.to_csv('AviationData_Cleaned.csv', index=False)
